# 02 — Task timing and condition taxonomy audit

Verifies contract timing constants, resolves canonical condition labels, maps omission slots, and writes timing/condition tables.

In [1]:
import sys
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _omission_run_common import (
    audit_area_mapping,
    audit_conditions,
    base_manifest,
    determine_time_base,
    project_root,
    resolve_nwb_path,
    resolve_run_root,
    resolve_nwb_sha256,
    run_nwb_audit,
    write_condition_table,
    write_json,
    write_timing_table,
    write_warnings,
)

start = time.time()
repo = project_root()
nwb_path = resolve_nwb_path(repo)
nwb_sha256 = resolve_nwb_sha256(nwb_path)
run_root = resolve_run_root(repo, nwb_path, nwb_sha256)
run_root.mkdir(parents=True, exist_ok=True)

def _run_timing_audit(nwbfile):
    area_status, area_warnings = audit_area_mapping(nwbfile)
    parser_status, condition_warnings, rows = audit_conditions(nwbfile)
    return {
        "area_mapping_status": area_status,
        "area_warnings": area_warnings,
        "condition_parser_status": parser_status,
        "condition_warnings": condition_warnings,
        "condition_rows": rows,
    }

warnings = []
audit_payload, open_error = run_nwb_audit(nwb_path, _run_timing_audit)
if open_error:
    warnings.append({"code": "PYNWB_OPEN_FAILED", "message": open_error})
    area_mapping_status = "not_applicable"
    condition_parser_status = "unresolved"
    time_base = "not_applicable"
    condition_rows = []
    notebook_status = "BLOCKED"
else:
    warnings.extend(audit_payload["area_warnings"])
    warnings.extend(audit_payload["condition_warnings"])
    area_mapping_status = audit_payload["area_mapping_status"]
    condition_parser_status = audit_payload["condition_parser_status"]
    condition_rows = audit_payload["condition_rows"]
    time_base = determine_time_base(condition_parser_status, nwb_open=True)
    notebook_status = "PASS" if condition_parser_status == "verified_with_evidence" else "BLOCKED"

timing_table_path = run_root / "tables" / "timing_table.csv"
condition_table_path = run_root / "tables" / "condition_table.csv"
write_timing_table(timing_table_path)
write_condition_table(condition_table_path, condition_rows)

manifest = base_manifest(
    notebook_id="02_task_timing_condition_audit",
    analysis_stage="timing_condition_audit",
    warnings_rel="warnings/02_warnings.json",
    runtime_seconds=time.time() - start,
    repo=repo,
    nwb_path=nwb_path,
    run_root=run_root,
    nwb_sha256=nwb_sha256,
    outputs=[
        "manifests/task_timing_condition_manifest.json",
        "tables/timing_table.csv",
        "tables/condition_table.csv",
        "warnings/02_warnings.json",
    ],
)
manifest["time_base"] = time_base
manifest["area_mapping_status"] = area_mapping_status
manifest["condition_parser_status"] = condition_parser_status
manifest["notebook_status"] = notebook_status

write_json(run_root / "manifests" / "task_timing_condition_manifest.json", manifest)
write_warnings(run_root / "warnings" / "02_warnings.json", warnings)

print(f"Task timing & condition audit status: {notebook_status}")
print("time_base:", time_base)
print("area_mapping_status:", area_mapping_status)
print("condition_parser_status:", condition_parser_status)
print("run_root:", run_root)
if notebook_status == "BLOCKED":
    raise RuntimeError("Notebook 02 blocked: timing/condition audit could not be verified.")